# Voxel AI — Sales Call Use Case Analysis
**Research & Analytics Intern Take-Home**

Two questions this notebook answers:
1. How much should we trust the current extraction output?
2. What non-safety opportunities are actually showing up in these calls?

**Approach:** Rule-based checks first (no LLM needed for deterministic issues), then Claude API for judgment-heavy tasks (label normalization, evidence quality scoring, business synthesis).

---
## 0. Setup & Imports

In [2]:
import json
import os
import re
import time
import pandas as pd
import anthropic
from collections import defaultdict
from dotenv import load_dotenv

# ── config ──────────────────────────────────────────────────────────────────
load_dotenv()

FOLDER = "safety-nonsafety/"
MODEL  = "claude-sonnet-4-5"

client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

files = [f for f in os.listdir(FOLDER) if f.endswith(".json")]
print(f"Files found: {len(files)}")

Files found: 99


In [3]:
# ── API key sanity check ─────────────────────────────────────────────────────
test = client.messages.create(
    model=MODEL,
    max_tokens=50,
    messages=[{"role": "user", "content": "Say hello in one sentence."}]
)
print("✓ API key working:", test.content[0].text)

✓ API key working: Hello, it's great to meet you!


---
## 1. Parse & Flatten — Build the Master DataFrame

One row = one extracted use case. We capture every field we will need downstream.

In [4]:
records = []

for fname in files:
    with open(f"{FOLDER}{fname}") as f:
        data = json.load(f)

    meeting_title = data.get("meeting_title", "Unknown")
    start_time    = data.get("start_time", "Unknown")
    extraction    = data.get("extraction", {})

    for bucket in ["safety_use_cases", "nonsafety_use_cases"]:
        for uc in extraction.get(bucket, []):
            evidence_list = uc.get("evidence", [])

            # collect individual quotes for dedup checks later
            quotes      = [e.get("quote", "").strip() for e in evidence_list]
            speakers    = [e.get("speaker", "") for e in evidence_list]
            timestamps  = [e.get("timestamp", None) for e in evidence_list]

            records.append({
                "file"           : fname,
                "meeting_title"  : meeting_title,
                "start_time"     : start_time,
                "bucket"         : bucket,
                "label"          : uc.get("label", "").strip(),
                "description"    : uc.get("description", "").strip(),
                "num_evidence"   : len(evidence_list),
                "evidence_quotes": " | ".join(quotes),
                "quotes_list"    : quotes,     # kept as list for exact matching
                "timestamps"     : timestamps, # kept as list for schema check
            })

df = pd.DataFrame(records)

print(f"Total use cases : {len(df)}")
print(f"  Safety        : {len(df[df.bucket == 'safety_use_cases'])}")
print(f"  Non-safety    : {len(df[df.bucket == 'nonsafety_use_cases'])}")
print(f"  Unique files  : {df.file.nunique()}")
df[["file", "meeting_title", "bucket", "label", "num_evidence"]].head(8)

Total use cases : 702
  Safety        : 471
  Non-safety    : 231
  Unique files  : 92


,file,meeting_title,bucket,label,num_evidence
0,31021d17-6f04-4dcd-97d6-d45d8e625c28.json,Delaney - Reconnect - North Works Operations +...,safety_use_cases,Pinpoint hazard hotspots with Heat Maps to dri...,9
1,31021d17-6f04-4dcd-97d6-d45d8e625c28.json,Delaney - Reconnect - North Works Operations +...,safety_use_cases,Boards for targeted safety monitoring and tren...,6
2,31021d17-6f04-4dcd-97d6-d45d8e625c28.json,Delaney - Reconnect - North Works Operations +...,safety_use_cases,Assign and track corrective safety actions wit...,9
3,31021d17-6f04-4dcd-97d6-d45d8e625c28.json,Delaney - Reconnect - North Works Operations +...,nonsafety_use_cases,Boards for targeted monitoring and trend detec...,6
4,31021d17-6f04-4dcd-97d6-d45d8e625c28.json,Delaney - Reconnect - North Works Operations +...,nonsafety_use_cases,Assign and track actions with owners and due d...,8
5,ab25c025-3c1d-4127-8784-51d4c905ea23.json,Stone Peak Manufacturing + Voxel Reconnect,safety_use_cases,Material handling and ergonomics risk detection,3
6,ab25c025-3c1d-4127-8784-51d4c905ea23.json,Stone Peak Manufacturing + Voxel Reconnect,safety_use_cases,Forklift intersection compliance (no-stop) mon...,2
7,ab25c025-3c1d-4127-8784-51d4c905ea23.json,Stone Peak Manufacturing + Voxel Reconnect,safety_use_cases,Walking under suspended loads detection,2


---
## 2. Rule-Based Quality Checks (No LLM Needed)

Deterministic issues don't need Claude — catching them programmatically is faster, cheaper, and more credible.

### 2a. Schema Integrity — Missing Timestamps

In [5]:
schema_issues = []

for _, row in df.iterrows():
    for i, ts in enumerate(row["timestamps"]):
        if not ts:  # None or empty string
            schema_issues.append({
                "file"  : row["file"],
                "label" : row["label"],
                "bucket": row["bucket"],
                "evidence_index": i,
                "issue" : "missing timestamp",
            })

schema_df = pd.DataFrame(schema_issues)
print(f"Evidence items with missing timestamps: {len(schema_df)}")
if len(schema_df):
    print(schema_df[["file", "label", "evidence_index"]].to_string(index=False))

Evidence items with missing timestamps: 222
                                     file                                                                                        label  evidence_index
31021d17-6f04-4dcd-97d6-d45d8e625c28.json                                 Pinpoint hazard hotspots with Heat Maps to drive mitigations               0
31021d17-6f04-4dcd-97d6-d45d8e625c28.json                                 Pinpoint hazard hotspots with Heat Maps to drive mitigations               1
31021d17-6f04-4dcd-97d6-d45d8e625c28.json                                 Pinpoint hazard hotspots with Heat Maps to drive mitigations               2
31021d17-6f04-4dcd-97d6-d45d8e625c28.json                                 Pinpoint hazard hotspots with Heat Maps to drive mitigations               3
31021d17-6f04-4dcd-97d6-d45d8e625c28.json                                 Pinpoint hazard hotspots with Heat Maps to drive mitigations               4
31021d17-6f04-4dcd-97d6-d45d8e625c28.json         

### 2b. Cross-Bucket Contamination — Same Quote in Both Buckets

In [6]:
# Explode so each row = one quote
quote_rows = []
for _, row in df.iterrows():
    for q in row["quotes_list"]:
        if q:
            quote_rows.append({"file": row["file"], "bucket": row["bucket"], "quote": q})

quote_df = pd.DataFrame(quote_rows)

# Find quotes that appear in BOTH buckets within the same file
contaminated = (
    quote_df.groupby(["file", "quote"])["bucket"]
    .nunique()
    .reset_index()
    .query("bucket > 1")
    .rename(columns={"bucket": "bucket_count"})
)

print(f"Quotes appearing in BOTH buckets (same file): {len(contaminated)}")
print(f"Affected files: {contaminated.file.nunique()}")
contaminated[["file", "quote"]].head(10)

Quotes appearing in BOTH buckets (same file): 207
Affected files: 48


,file,quote
36,03a609d7-bd48-48aa-a708-650afb1d3a58.json,"No speeding, no idling, suspended load when th..."
38,03a609d7-bd48-48aa-a708-650afb1d3a58.json,One of the lower-hanging opportunities is redu...
44,03a609d7-bd48-48aa-a708-650afb1d3a58.json,They had something on the floor that led to a ...
61,03a609d7-bd48-48aa-a708-650afb1d3a58.json,we look at their historical incident data to u...
78,0511afa9-94d6-4a22-a384-11efbef34dfe.json,if you perform your typical safety observation...
79,0511afa9-94d6-4a22-a384-11efbef34dfe.json,"stores run with less and less team members, so..."
84,0511afa9-94d6-4a22-a384-11efbef34dfe.json,what we can't do is put something out there th...
96,0672b03e-87a3-410a-977c-35e1d46245c5.json,Think broadly about what can go wrong in custo...
98,0672b03e-87a3-410a-977c-35e1d46245c5.json,We're exploring front-of-store retail use case...
105,096964ad-8b87-401e-933c-984c56c0da56.json,Here's what the platform told us.


In [7]:
# Flag the affected rows in the main df
contaminated_files = set(contaminated["file"].tolist())
df["cross_bucket_flag"] = df["file"].isin(contaminated_files)

print("Files with cross-bucket contamination:")
for f in contaminated_files:
    title = df[df.file == f]["meeting_title"].iloc[0]
    print(f"  {f[:8]}... — {title}")

Files with cross-bucket contamination:
  3a43e278... — (ext) Voxel | Falcon Point Fabrication (LP brief)
  276c425a... — Quarry Flow Logistics + Voxel Intro Call
  ab25c025... — Stone Peak Manufacturing + Voxel Reconnect
  e4b2f981... — Voxel Safety Intro: Copper Forge Industrial
  5e295dd9... — Prairie Bridge Systems / Voxel Monthly Sync
  f2575681... — Iron Flow Manufacturing + Voxel - Reconnect
  aa7a4873... — Blue Peak Services POC Week 2 Review
  684871c9... — Prime Signal Materials / Voxel Sync 
  9f23bf36... — Bridge Mill Transport / Voxel Bi-Weekly Sync
  0672b03e... — Signal Grid Works / Voxel Bi-Weekly Sync
  765f10aa... — Forge Axis Solutions / Voxel Monthly Sync
  03a609d7... — Orion Forge Industrial // Voxel
  71760067... — Blue Base Supply / Voxel:  Discovery discussion and platform overview
  cc1e178e... — Cobalt Plant Industrial / Voxel pilot discussion
  6a626eda... — Riley/Cleo Check In
  512af163... — Voxel - Cobalt Ridge Fabrication
  249676e3... — Kodiak Point Fabr

### 2c. Admin Noise Candidates — Files with Zero Safety Use Cases

In [8]:
file_summary = df.groupby("file").agg(
    meeting_title    = ("meeting_title", "first"),
    has_safety       = ("bucket", lambda x: "safety_use_cases" in x.values),
    has_nonsafety    = ("bucket", lambda x: "nonsafety_use_cases" in x.values),
    total_use_cases  = ("label", "count"),
).reset_index()

# No safety at all — likely admin/commercial calls
noise_candidates = file_summary[
    (~file_summary["has_safety"]) & (file_summary["has_nonsafety"])
]

print(f"Files with ONLY nonsafety extractions (admin noise candidates): {len(noise_candidates)}")
noise_candidates[["file", "meeting_title", "total_use_cases"]]

Files with ONLY nonsafety extractions (admin noise candidates): 4


,file,meeting_title,total_use_cases
11,1ffac178-0eee-4534-8fac-8babb3831770.json,Voxel Ai + Iron Harbor Fleet Harbor Manufacturing,2
52,9ef26ef2-778c-4fdd-9a77-d1b07e9d2840.json,Cobalt Plant Industrial + Voxel IT/Network call,3
58,a7293f69-af12-41b7-bfa9-a904e3bffa11.json,Meridian Works Operations MMS / Voxel - site s...,1
69,c3ceb91a-e15d-4326-8a39-cfdb26dde294.json,Summit Span Solutions / Voxel Sync,5


In [9]:
# Mark noise files in master df — we'll exclude them from opportunity ranking
noise_files = set(noise_candidates["file"].tolist())
df["admin_noise_flag"] = df["file"].isin(noise_files)

print(f"Use cases in noise files (will be excluded from ranking): {df.admin_noise_flag.sum()}")

Use cases in noise files (will be excluded from ranking): 11


### 2d. Shared Evidence — Same Quotes Recycled Across Two Use Cases (Same File, Same Bucket)

In [10]:
# Within same file + bucket, find quotes appearing under >1 label
shared_evidence = (
    quote_df.merge(
        df[["file", "bucket", "label", "quotes_list"]].explode("quotes_list")
          .rename(columns={"quotes_list": "quote"}),
        on=["file", "bucket", "quote"]
    )
    .groupby(["file", "bucket", "quote"])["label"]
    .nunique()
    .reset_index()
    .query("label > 1")
    .rename(columns={"label": "label_count"})
)

print(f"Quotes recycled across multiple labels (same file, same bucket): {len(shared_evidence)}")
shared_evidence.head(10)

Quotes recycled across multiple labels (same file, same bucket): 20


,file,bucket,quote,label_count
293,20a3a151-a51a-48ed-b23d-203409be2306.json,safety_use_cases,"coming out of the onsite, PIT to pit, ergonomi...",2
429,2fb0bc4b-0c18-4eae-8ff8-988e11af0d41.json,safety_use_cases,Let's say there was a spill on the floor and e...,2
490,37033189-bb15-419d-bd12-d9c07c44acbd.json,safety_use_cases,Forklift-to-Person Failure.,2
615,44dcf03a-68a7-4736-9e66-b16b6462bd2b.json,safety_use_cases,"We have improper bends, overreachings, such as...",2
640,45f3120e-3169-4083-8b5c-c97d0f20cb0c.json,safety_use_cases,"priority one is identifying ergonomic risk, ne...",2
667,45f8972e-4307-4256-adf1-45ca7a833e72.json,safety_use_cases,"Yeah, I mean, that's exactly where ergonomics,...",2
678,46c2fe16-50d6-44a5-a327-2443fc6f1ba7.json,safety_use_cases,"Material handling, ergonomics, vehicle movemen...",2
741,512af163-fb6a-447d-a767-f0cbb22fca95.json,nonsafety_use_cases,a heat map is extremely powerful when you're l...,2
1684,b7320572-5b94-4ad1-a61f-1d9fe3310b67.json,safety_use_cases,I usually take a look at the Wednesday and Thu...,2
1722,bc645b88-89e5-4865-8782-5e17b6a20c20.json,safety_use_cases,"They almost collided, and, you know, this may ...",2


### 2e. Transcription Artifact Check — Known Mistranscriptions

In [11]:
# 'person to pet' is a known transcription error for 'PIT' (Powered Industrial Truck)
# Flag any quote containing suspicious phrases
SUSPECT_PHRASES = [
    r"person to pet",      # should be 'person to PIT'
    r"pit to pet",         # same issue
    r"\bget some vintage\b",  # likely 'get some video' or 'get some footage'
]

transcription_flags = []
for _, row in df.iterrows():
    for q in row["quotes_list"]:
        for pattern in SUSPECT_PHRASES:
            if re.search(pattern, q, re.IGNORECASE):
                transcription_flags.append({
                    "file"   : row["file"],
                    "label"  : row["label"],
                    "quote"  : q,
                    "pattern": pattern,
                })

trans_df = pd.DataFrame(transcription_flags)
print(f"Likely transcription artifacts found: {len(trans_df)}")
if len(trans_df):
    print(trans_df[["file", "label", "quote"]].to_string(index=False))

Likely transcription artifacts found: 7
                                     file                                                                                  label                                                                                                                                                                                                                            quote
d3934fd4-3e44-405b-bcc4-d3556dd6d20c.json                                     PIT/pedestrian proximity monitoring in drive lanes                                                                                                                                                                                 improper bend, person to pet, overreaching, yes.
d3934fd4-3e44-405b-bcc4-d3556dd6d20c.json                                 Ergonomic risk detection (improper bend, overreaching)                                                                                                                            

### 2f. Rule-Based Summary — Quality Issues Dashboard

In [12]:
print("=" * 55)
print("PIPELINE QUALITY AUDIT — RULE-BASED CHECKS")
print("=" * 55)
print(f"Total files          : {len(files)}")
print(f"Total use cases      : {len(df)}")
print()
print(f"[1] Missing timestamps     : {len(schema_df)} evidence items")
print(f"[2] Cross-bucket contamination: {len(contaminated_files)} files, {len(contaminated)} shared quotes")
print(f"[3] Admin noise candidates : {len(noise_candidates)} files ({len(noise_candidates)/len(files)*100:.1f}% of corpus)")
print(f"[4] Recycled evidence      : {len(shared_evidence)} quotes used across multiple labels")
print(f"[5] Transcription artifacts: {len(trans_df)} suspect quotes")
print("=" * 55)

PIPELINE QUALITY AUDIT — RULE-BASED CHECKS
Total files          : 99
Total use cases      : 702

[1] Missing timestamps     : 222 evidence items
[2] Cross-bucket contamination: 48 files, 207 shared quotes
[3] Admin noise candidates : 4 files (4.0% of corpus)
[4] Recycled evidence      : 20 quotes used across multiple labels
[5] Transcription artifacts: 7 suspect quotes


---
## 3. Claude API — Label Normalization & Clustering (Split by Bucket)

Labels are split into safety and nonsafety batches to stay within token limits.
Cross-bucket contamination is handled separately in Step 2 (rule-based).
Claude's job here is purely grouping semantically similar labels together.

In [13]:
# Build deduplicated label list with bucket info
label_list = (
    df[["label", "bucket"]]
    .drop_duplicates()
    .sort_values("bucket")
    .reset_index(drop=True)
)

label_text = "\n".join(
    [f"{i+1}. [{row.bucket.replace('_use_cases','')}] {row.label}"
     for i, row in label_list.iterrows()]
)

print(f"Unique labels to cluster: {len(label_list)}")
print(label_text[:800], "\n...")

Unique labels to cluster: 687
1. [nonsafety] Reduce shrink by detecting product damage from PIT impacts
2. [nonsafety] Ongoing partnership and customer success support
3. [nonsafety] Reduce implementation/training burden and false positives
4. [nonsafety] Run a trial before broader rollout
5. [nonsafety] Restore dashboard deep links in daily incident email
6. [nonsafety] Integration with incident-management and reporting systems
7. [nonsafety] Role‑based boards and dashboard customization for MODs
8. [nonsafety] Planning for multi-site rollout requirements
9. [nonsafety] Responsive customer support and troubleshooting
10. [nonsafety] Action workflow and accountability tracking
11. [nonsafety] Optimize pallet stack heights and segregate pallet types to improve productivity
12. [nonsafety] Operational analytics: monitor  
...


In [18]:
# ── helper: build prompt for one bucket ──────────────────────────────────────
def build_normalization_prompt(labels_df, bucket_name):
    label_text_local = "\n".join(
        [f"{i+1}. {row.label}" for i, row in labels_df.reset_index(drop=True).iterrows()]
    )
    return f"""You are analyzing use case labels extracted from enterprise sales call transcripts 
for Voxel, an AI workplace safety company that uses computer vision on existing security cameras.

Below is a numbered list of [{bucket_name}] use case labels.

Your tasks:
1. Group labels that refer to the same underlying concept (even if worded differently).
2. Assign each group a short, clean canonical_name (3-6 words max).
3. Flag generic_flag: true if the label is too vague to be a real product use case 
   (e.g. pure admin/billing topics, one-off customer anecdotes, unrelated technology mentions).
4. generic_reason: one sentence if generic_flag is true, else null.

Return ONLY valid JSON, no preamble, no markdown fences:
{{
  "clusters": [
    {{
      "canonical_name": "...",
      "member_labels": ["..."],
      "generic_flag": true/false,
      "generic_reason": "one sentence if generic_flag is true, else null"
    }}
  ]
}}

Labels:
{label_text_local}
"""

# ── split labels by bucket ────────────────────────────────────────────────────
safety_labels    = label_list[label_list.bucket == "safety_use_cases"]
nonsafety_labels = label_list[label_list.bucket == "nonsafety_use_cases"]

# ── Call 1: safety ────────────────────────────────────────────────────────────
print("Sending label normalization request to Claude [safety]...")
safety_response = client.messages.create(
    model=MODEL,
    max_tokens=16000,
    messages=[{"role": "user", "content": build_normalization_prompt(safety_labels, "safety")}]
)
raw_safety = safety_response.content[0].text
if safety_response.stop_reason != "end_turn":
    print(f"⚠ WARNING: Safety response stopped due to '{safety_response.stop_reason}' — may be truncated")
    print(f"  Characters returned: {len(raw_safety)}")
    print(f"  Last 200 chars: ...{raw_safety[-200:]}")
else:
    print(f"✓ Safety response complete. Characters: {len(raw_safety)}")

# ── Call 2: nonsafety ─────────────────────────────────────────────────────────
print("Sending label normalization request to Claude [nonsafety]...")
nonsafety_response = client.messages.create(
    model=MODEL,
    max_tokens=16000,
    messages=[{"role": "user", "content": build_normalization_prompt(nonsafety_labels, "nonsafety")}]
)
raw_nonsafety = nonsafety_response.content[0].text
if nonsafety_response.stop_reason != "end_turn":
    print(f"⚠ WARNING: Nonsafety response stopped due to '{nonsafety_response.stop_reason}' — may be truncated")
    print(f"  Characters returned: {len(raw_nonsafety)}")
    print(f"  Last 200 chars: ...{raw_nonsafety[-200:]}")
else:
    print(f"✓ Nonsafety response complete. Characters: {len(raw_nonsafety)}")

Sending label normalization request to Claude [safety]...
✓ Safety response complete. Characters: 39852
Sending label normalization request to Claude [nonsafety]...
✓ Nonsafety response complete. Characters: 26983


In [22]:
!pip install json-repair -q

In [23]:
from json_repair import repair_json

def parse_clusters(raw, bucket_name):
    clean = re.sub(r"```json|```", "", raw).strip()
    match = re.search(r'\{.*\}', clean, re.DOTALL)
    if not match:
        raise ValueError(f"No JSON found in {bucket_name} response")
    
    # json_repair handles unescaped quotes, trailing commas, all of it
    repaired = repair_json(match.group())
    data = json.loads(repaired)

    df = pd.DataFrame(data["clusters"])
    df["bucket"] = bucket_name
    print(f"  ✓ {bucket_name}: {len(df)} clusters parsed")
    return df

# ── re-run parse ──────────────────────────────────────────────────────────────
safety_clusters_df    = parse_clusters(raw_safety, "safety")
nonsafety_clusters_df = parse_clusters(raw_nonsafety, "nonsafety")

# ── contamination: programmatic set intersection ──────────────────────────────
safety_names       = set(safety_clusters_df["canonical_name"].str.lower())
nonsafety_names    = set(nonsafety_clusters_df["canonical_name"].str.lower())
contaminated_names = safety_names & nonsafety_names

safety_clusters_df["contamination_flag"] = (
    safety_clusters_df["canonical_name"].str.lower().isin(contaminated_names)
)
nonsafety_clusters_df["contamination_flag"] = (
    nonsafety_clusters_df["canonical_name"].str.lower().isin(contaminated_names)
)

clusters_df = pd.concat([safety_clusters_df, nonsafety_clusters_df], ignore_index=True)

print(f"\nTotal clusters identified  : {len(clusters_df)}")
print(f"  Safety clusters          : {len(safety_clusters_df)}")
print(f"  Nonsafety clusters       : {len(nonsafety_clusters_df)}")
print(f"Contaminated (both buckets): {clusters_df.contamination_flag.sum()}")
print(f"Flagged as generic/noise   : {clusters_df.generic_flag.sum()}")
print()
clusters_df[["canonical_name", "bucket", "contamination_flag", "generic_flag"]].head(20)

  ✓ safety: 64 clusters parsed
  ✓ nonsafety: 71 clusters parsed

Total clusters identified  : 135
  Safety clusters          : 64
  Nonsafety clusters       : 71
Contaminated (both buckets): 4
Flagged as generic/noise   : 22



,canonical_name,bucket,contamination_flag,generic_flag
0,Ergonomic Risk Detection,safety,False,False
1,PIT-Pedestrian Proximity Monitoring,safety,False,False
2,PIT-to-PIT Proximity Monitoring,safety,False,False
3,PPE Compliance Monitoring,safety,True,False
4,Spill Detection and Monitoring,safety,False,False
5,Working at Heights Detection,safety,False,False
6,Obstruction and Egress Monitoring,safety,False,False
7,Forklift Speed Monitoring,safety,False,False
8,Intersection Stop Compliance,safety,False,False
9,No-Pedestrian Zone Enforcement,safety,False,False


In [24]:
# Build lookup: original label -> canonical_name
label_to_canonical = {}
for _, cluster in clusters_df.iterrows():
    for member in cluster["member_labels"]:
        label_to_canonical[member] = cluster["canonical_name"]

df["canonical"] = df["label"].map(label_to_canonical).fillna(df["label"])

# Also bring in the generic and contamination flags per row
label_to_generic       = {}
label_to_contamination = {}
for _, cluster in clusters_df.iterrows():
    for member in cluster["member_labels"]:
        label_to_generic[member]       = cluster["generic_flag"]
        label_to_contamination[member] = cluster["contamination_flag"]

df["generic_flag"]       = df["label"].map(label_to_generic).fillna(False)
df["llm_contamination"]  = df["label"].map(label_to_contamination).fillna(False)

print("Canonical names merged into master df.")
df[["label", "canonical", "generic_flag", "llm_contamination"]].head(10)

Canonical names merged into master df.


/var/folders/tj/j_j3x8fd05xfhgf4yvjjphsc0000gn/T/ipykernel_39887/1089681934.py:17: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["generic_flag"]       = df["label"].map(label_to_generic).fillna(False)
/var/folders/tj/j_j3x8fd05xfhgf4yvjjphsc0000gn/T/ipykernel_39887/1089681934.py:18: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["llm_contamination"]  = df["label"].map(label_to_contamination).fillna(False)


,label,canonical,generic_flag,llm_contamination
0,Pinpoint hazard hotspots with Heat Maps to dri...,Safety Heatmaps and Hotspot Analysis,False,False
1,Boards for targeted safety monitoring and tren...,Boards for targeted safety monitoring and tren...,False,False
2,Assign and track corrective safety actions wit...,Actions Workflow and Coaching,False,False
3,Boards for targeted monitoring and trend detec...,Dashboard and UI Customization,False,False
4,Assign and track actions with owners and due d...,Action Tracking and Accountability,False,False
5,Material handling and ergonomics risk detection,Ergonomic Risk Detection,False,False
6,Forklift intersection compliance (no-stop) mon...,Intersection Stop Compliance,False,False
7,Walking under suspended loads detection,Suspended Load Hazard Detection,False,False
8,Working at heights without harness detection,Working at Heights Detection,False,False
9,PPE compliance and laceration prevention,PPE Compliance Monitoring,False,True


## Creating Backup, in case the notbook is restarted

In [25]:
import pickle

# Save the raw LLM responses — most important, these cost money to regenerate
with open("raw_safety.txt", "w") as f:
    f.write(raw_safety)
with open("raw_nonsafety.txt", "w") as f:
    f.write(raw_nonsafety)

# Save the processed dataframes
df.to_csv("master_df.csv", index=False)
clusters_df.to_csv("clusters_df.csv", index=False)

# Save everything as pickle too (preserves list columns like quotes_list)
with open("session_checkpoint.pkl", "wb") as f:
    pickle.dump({
        "df": df,
        "clusters_df": clusters_df,
        "raw_safety": raw_safety,
        "raw_nonsafety": raw_nonsafety,
    }, f)

print("✓ All results saved to disk — safe to close")

✓ All results saved to disk — safe to close


## Resume by loading datasets below

In [26]:
import pickle

with open("session_checkpoint.pkl", "rb") as f:
    checkpoint = pickle.load(f)

df          = checkpoint["df"]
clusters_df = checkpoint["clusters_df"]
raw_safety  = checkpoint["raw_safety"]
raw_nonsafety = checkpoint["raw_nonsafety"]

print("✓ Session restored")
print(f"  df shape: {df.shape}")
print(f"  clusters: {len(clusters_df)}")

✓ Session restored
  df shape: (702, 15)
  clusters: 135


---
## 4. Claude API — Evidence Quality Scoring

For each use case: does the evidence actually justify the label? We score 1-5 on two dimensions.

In [27]:
def score_evidence(row):
    """Score a single use case. Returns dict with scores and issue note."""
    prompt = f"""You are auditing an AI extraction pipeline for Voxel, a computer vision safety company.

A use case was extracted from a sales call transcript:

Label       : {row['label']}
Description : {row['description']}
Bucket      : {row['bucket']}
Evidence    : {row['evidence_quotes'][:1200]}  

Score on two dimensions (integers 1-5):

evidence_support (1-5):
  5 = multiple clear quotes that directly prove this label
  3 = partial support, some ambiguity
  1 = little or no connection between quotes and label

label_accuracy (1-5):
  5 = label is precise, specific, and correctly categorized
  3 = label is roughly correct but vague or slightly off
  1 = label overstates, misrepresents, or is wrongly bucketed

Also:
- issue: one-sentence note if EITHER score < 4, else null
- off_topic: true if the content is unrelated to Voxel's camera AI platform (e.g. wearables, billing admin, competitor tech)

Return ONLY valid JSON, no markdown:
{{"evidence_support": int, "label_accuracy": int, "issue": "..." or null, "off_topic": true/false}}
"""
    try:
        resp = client.messages.create(
            model=MODEL,
            max_tokens=250,
            messages=[{"role": "user", "content": prompt}]
        )
        raw = resp.content[0].text
        clean = re.sub(r"```json|```", "", raw).strip()
        match = re.search(r"\{.*\}", clean, re.DOTALL)
        return json.loads(match.group())
    except Exception as e:
        return {"evidence_support": None, "label_accuracy": None,
                "issue": f"parse error: {str(e)}", "off_topic": None}


print(f"Scoring {len(df)} use cases — this takes a few minutes...")
scores = []
for i, row in df.iterrows():
    scores.append(score_evidence(row))
    if (i + 1) % 25 == 0:
        print(f"  {i+1}/{len(df)} scored...")
    time.sleep(0.3)  # light rate-limit buffer

print("Done.")

Scoring 702 use cases — this takes a few minutes...
  25/702 scored...
  50/702 scored...
  75/702 scored...
  100/702 scored...
  125/702 scored...
  150/702 scored...
  175/702 scored...
  200/702 scored...
  225/702 scored...
  250/702 scored...
  275/702 scored...
  300/702 scored...
  325/702 scored...
  350/702 scored...
  375/702 scored...
  400/702 scored...
  425/702 scored...
  450/702 scored...
  475/702 scored...
  500/702 scored...
  525/702 scored...
  550/702 scored...
  575/702 scored...
  600/702 scored...
  625/702 scored...
  650/702 scored...
  675/702 scored...
  700/702 scored...
Done.


In [28]:
scores_df = pd.DataFrame(scores)
df = pd.concat([df.reset_index(drop=True), scores_df], axis=1)

# Combined quality score
df["quality_score"] = (
    df["evidence_support"].fillna(0) + df["label_accuracy"].fillna(0)
) / 2

print("Evidence scores merged.")
print(f"Avg evidence_support : {df.evidence_support.mean():.2f} / 5")
print(f"Avg label_accuracy   : {df.label_accuracy.mean():.2f} / 5")
print(f"Off-topic rows       : {df.off_topic.sum()}")
print()
print("Weakest extractions (quality_score < 3):")
weak = df[df.quality_score < 3][["file", "label", "evidence_support", "label_accuracy", "issue"]]
print(weak.to_string(index=False))

Evidence scores merged.
Avg evidence_support : 4.75 / 5
Avg label_accuracy   : 4.34 / 5
Off-topic rows       : 11

Weakest extractions (quality_score < 3):
                                     file                                                                       label  evidence_support  label_accuracy                                                                                                                                                                                                                                                                                                                                      issue
3a43e278-01b9-4daf-9ee7-978f585cf978.json                                      Machine shutdown use case (operations)                 2               3                                                                                                                                                          Evidence only vaguely mentions machine shutdown as 'one 

## Saving Datasets As Backup

In [29]:
# Save raw LLM responses
with open("raw_safety.txt", "w") as f:
    f.write(raw_safety)
with open("raw_nonsafety.txt", "w") as f:
    f.write(raw_nonsafety)

# Save all dataframes as CSV
df.to_csv("master_df_scored.csv", index=False)
clusters_df.to_csv("clusters_df.csv", index=False)

# Save full session as pickle (preserves list columns + all objects)
with open("session_checkpoint_scored.pkl", "wb") as f:
    pickle.dump({
        "df"            : df,
        "clusters_df"   : clusters_df,
        "scores"        : scores,
        "scores_df"     : scores_df,
        "raw_safety"    : raw_safety,
        "raw_nonsafety" : raw_nonsafety,
        "label_to_canonical"    : label_to_canonical,
        "label_to_generic"      : label_to_generic,
        "label_to_contamination": label_to_contamination,
    }, f)

print("✓ Checkpoint saved — safe to close")
print(f"  df shape        : {df.shape}")
print(f"  clusters        : {len(clusters_df)}")
print(f"  scores          : {len(scores)}")

✓ Checkpoint saved — safe to close
  df shape        : (702, 20)
  clusters        : 135
  scores          : 702


---
## 5. Filter & Rank Non-Safety Opportunities

Now we have a clean, annotated dataset. Apply filters before ranking so the results are trustworthy.

In [30]:
nonsafety_raw = df[df.bucket == "nonsafety_use_cases"].copy()
print(f"Raw nonsafety rows: {len(nonsafety_raw)}")

# Apply quality filters
nonsafety_clean = nonsafety_raw[
    (~nonsafety_raw["admin_noise_flag"])   # exclude pure admin files
    & (~nonsafety_raw["generic_flag"])     # exclude generic/noise labels
    & (~nonsafety_raw["off_topic"].fillna(False))  # exclude off-topic
    & (nonsafety_raw["evidence_support"].fillna(0) >= 2)  # minimum evidence bar
].copy()

print(f"After quality filters: {len(nonsafety_clean)} rows")
print(f"Excluded: {len(nonsafety_raw) - len(nonsafety_clean)} rows")

Raw nonsafety rows: 231
After quality filters: 193 rows
Excluded: 38 rows


In [31]:
# What actually got excluded and why
excluded = nonsafety_raw[~nonsafety_raw.index.isin(nonsafety_clean.index)].copy()

print(f"Excluded by admin_noise_flag : {(excluded.admin_noise_flag).sum()}")
print(f"Excluded by generic_flag     : {(excluded.generic_flag).sum()}")
print(f"Excluded by off_topic        : {(excluded.off_topic.fillna(False)).sum()}")
print(f"Excluded by weak evidence    : {(excluded.evidence_support.fillna(0) < 2).sum()}")
print()
print("Note: rows can be excluded by multiple flags simultaneously")

Excluded by admin_noise_flag : 11
Excluded by generic_flag     : 32
Excluded by off_topic        : 9
Excluded by weak evidence    : 0

Note: rows can be excluded by multiple flags simultaneously


In [32]:
# Rank by: how many distinct calls mention this canonical theme + avg evidence quality
opportunity_ranking = (
    nonsafety_clean.groupby("canonical")
    .agg(
        call_count           = ("file", "nunique"),
        avg_evidence_support = ("evidence_support", "mean"),
        avg_label_accuracy   = ("label_accuracy", "mean"),
        example_labels       = ("label", lambda x: list(x.unique())[:3]),
        sample_quotes        = ("evidence_quotes", lambda x: x.iloc[0][:300]),
    )
    .sort_values(["call_count", "avg_evidence_support"], ascending=False)
    .reset_index()
)

print("Top non-safety opportunities (by call frequency + evidence quality):")
print()
opportunity_ranking[
    ["canonical", "call_count", "avg_evidence_support", "avg_label_accuracy"]
].head(10).to_string(index=False)

Top non-safety opportunities (by call frequency + evidence quality):



'                          canonical  call_count  avg_evidence_support  avg_label_accuracy\n Action Tracking and Accountability          20              4.863636            3.590909\n           Door Duration Monitoring          11              4.363636            3.818182\n     Dashboard and UI Customization           8              5.000000            4.222222\n     Safety Analytics and Reporting           8              5.000000            2.500000\n System Integration and Data Export           8              4.625000            3.875000\nProduct Damage and Shrink Reduction           8              4.222222            3.444444\n Restricted Area and Access Control           7              5.000000            3.625000\nWarehouse Productivity Optimization           7              4.428571            3.285714\n    Report Export and Subscriptions           6              4.750000            3.875000\n       Intervention Impact Tracking           5              5.000000            3.000000

In [33]:
# Rank by: how many distinct calls mention this canonical theme + avg evidence quality
opportunity_ranking = (
    nonsafety_clean.groupby("canonical")
    .agg(
        call_count           = ("file", "nunique"),
        avg_evidence_support = ("evidence_support", "mean"),
        avg_label_accuracy   = ("label_accuracy", "mean"),
        example_labels       = ("label", lambda x: list(x.unique())[:3]),
        sample_quotes        = ("evidence_quotes", lambda x: x.iloc[0][:300]),
    )
    .sort_values(["call_count", "avg_evidence_support"], ascending=False)
    .reset_index()
)

# ── display top 10 nicely ─────────────────────────────────────────────────────
display_df = opportunity_ranking[
    ["canonical", "call_count", "avg_evidence_support", "avg_label_accuracy"]
].head(10).copy()

display_df.columns = ["Canonical Theme", "# Calls", "Avg Evidence (1-5)", "Avg Label Acc (1-5)"]
display_df["Avg Evidence (1-5)"] = display_df["Avg Evidence (1-5)"].round(2)
display_df["Avg Label Acc (1-5)"] = display_df["Avg Label Acc (1-5)"].round(2)
display_df = display_df.reset_index(drop=True)
display_df.index += 1  # start ranking at 1

display(display_df.style
    .set_caption("Top 10 Non-Safety Opportunities — Ranked by Call Frequency + Evidence Quality")
    .background_gradient(subset=["# Calls"], cmap="Greens")
    .background_gradient(subset=["Avg Evidence (1-5)"], cmap="Blues")
    .set_properties(**{"text-align": "left"})
    .set_table_styles([
        {"selector": "caption", "props": [("font-size", "13px"), ("font-weight", "bold"), ("padding-bottom", "8px")]},
        {"selector": "th", "props": [("font-size", "12px"), ("text-align", "left"), ("padding", "6px 12px")]},
        {"selector": "td", "props": [("padding", "5px 12px"), ("font-size", "12px")]},
    ])
)

,Canonical Theme,# Calls,Avg Evidence (1-5),Avg Label Acc (1-5)
1,Action Tracking and Accountability,20,4.860000,3.590000
2,Door Duration Monitoring,11,4.360000,3.820000
3,Dashboard and UI Customization,8,5.000000,4.220000
4,Safety Analytics and Reporting,8,5.000000,2.500000
5,System Integration and Data Export,8,4.620000,3.880000
6,Product Damage and Shrink Reduction,8,4.220000,3.440000
7,Restricted Area and Access Control,7,5.000000,3.620000
8,Warehouse Productivity Optimization,7,4.430000,3.290000
9,Report Export and Subscriptions,6,4.750000,3.880000
10,Intervention Impact Tracking,5,5.000000,3.000000


In [61]:
# ── opportunity ranking with audit verdicts ───────────────────────────────────
opportunity_ranking = (
    nonsafety_clean.groupby("canonical")
    .agg(
        call_count           = ("file", "nunique"),
        avg_evidence_support = ("evidence_support", "mean"),
        avg_label_accuracy   = ("label_accuracy", "mean"),
        example_labels       = ("label", lambda x: list(x.unique())[:3]),
        sample_quotes        = ("evidence_quotes", lambda x: x.iloc[0][:300]),
    )
    .sort_values(["call_count", "avg_evidence_support"], ascending=False)
    .reset_index()
)

# ── audit verdicts from Layer 3 review ───────────────────────────────────────
verdicts = {
    "Action Tracking and Accountability" : "DISQUALIFIED",
    "Door Duration Monitoring"           : "VALID — Real #1",
    "Dashboard and UI Customization"     : "VALID — Real #2",
    "Safety Analytics and Reporting"     : "BORDERLINE",
    "System Integration and Data Export" : "VALID — Real #3",
    "Product Damage and Shrink Reduction": "VALID",
    "Restricted Area and Access Control" : "VALID",
    "Warehouse Productivity Optimization": "MIXED",
    "Report Export and Subscriptions"    : "MIXED",
    "Intervention Impact Tracking"       : "BORDERLINE",
}

display_df = opportunity_ranking[
    ["canonical", "call_count", "avg_evidence_support", "avg_label_accuracy"]
].head(10).copy()

display_df.columns = ["Canonical Theme", "# Calls", "Avg Evidence (1-5)", "Avg Label Acc (1-5)"]
display_df["Avg Evidence (1-5)"]  = display_df["Avg Evidence (1-5)"].round(2)
display_df["Avg Label Acc (1-5)"] = display_df["Avg Label Acc (1-5)"].round(2)
display_df["Audit Verdict"]       = display_df["Canonical Theme"].map(verdicts).fillna("")
display_df = display_df.reset_index(drop=True)
display_df.index += 1

def color_verdict(val):
    if "DISQUALIFIED" in str(val):
        return "background-color: #FEE2E2; color: #991B1B; font-weight: bold"
    elif "Real" in str(val):
        return "background-color: #D1FAE5; color: #065F46; font-weight: bold"
    elif "BORDERLINE" in str(val):
        return "background-color: #FEF3C7; color: #92400E"
    elif "MIXED" in str(val):
        return "background-color: #F3F4F6; color: #374151"
    elif "VALID" in str(val):
        return "background-color: #D1FAE5; color: #065F46"
    return ""

display(display_df.style
    .set_caption("Top 10 Non-Safety Opportunities — Raw Ranking + Audit Verdict")
    .background_gradient(subset=["# Calls"], cmap="Greens")
    .background_gradient(subset=["Avg Evidence (1-5)"], cmap="Blues")
    .applymap(color_verdict, subset=["Audit Verdict"])
    .format({"Avg Evidence (1-5)": "{:.2f}", "Avg Label Acc (1-5)": "{:.2f}"})  # ← this line
    .set_properties(**{"text-align": "left"})
    .set_table_styles([
        {"selector": "caption",
         "props": [("font-size", "13px"), ("font-weight", "bold"), ("padding-bottom", "8px")]},
        {"selector": "th",
         "props": [("font-size", "12px"), ("text-align", "left"), ("padding", "6px 12px")]},
        {"selector": "td",
         "props": [("padding", "5px 12px"), ("font-size", "12px")]},
    ])
)

/var/folders/tj/j_j3x8fd05xfhgf4yvjjphsc0000gn/T/ipykernel_39887/1750837604.py:57: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  .applymap(color_verdict, subset=["Audit Verdict"])


,Canonical Theme,# Calls,Avg Evidence (1-5),Avg Label Acc (1-5),Audit Verdict
1,Action Tracking and Accountability,20,4.86,3.59,DISQUALIFIED
2,Door Duration Monitoring,11,4.36,3.82,VALID — Real #1
3,Dashboard and UI Customization,8,5.00,4.22,VALID — Real #2
4,Safety Analytics and Reporting,8,5.00,2.50,BORDERLINE
5,System Integration and Data Export,8,4.62,3.88,VALID — Real #3
6,Product Damage and Shrink Reduction,8,4.22,3.44,VALID
7,Restricted Area and Access Control,7,5.00,3.62,VALID
8,Warehouse Productivity Optimization,7,4.43,3.29,MIXED
9,Report Export and Subscriptions,6,4.75,3.88,MIXED
10,Intervention Impact Tracking,5,5.00,3.00,BORDERLINE


---
## 6. Claude API — Business Memo Synthesis

Feed the ranked data back to Claude for a CTO-ready narrative. This is the synthesis layer — not the analysis.

In [64]:
top_opps = opportunity_ranking.head(6).to_dict(orient="records")

# Clean up for JSON serialization
for opp in top_opps:
    opp["avg_evidence_support"] = round(float(opp["avg_evidence_support"]), 2)
    opp["avg_label_accuracy"]   = round(float(opp["avg_label_accuracy"]), 2)

# ── NOTE: Action Tracking and Accountability was disqualified in Layer 3 audit ──
# It ranked #1 by frequency but all 22 member labels are safety workflow content.
# The corrected top 3 based on audit: Door Duration Monitoring,
# Dashboard and UI Customization, System Integration and Data Export.

memo_prompt = f"""You are a senior analyst at Voxel, an AI workplace safety company.
Voxel's platform uses existing security cameras with computer vision to detect safety risks
in warehouses, manufacturing plants, and distribution centers.

Below are the top non-safety use cases extracted from {len(files)} enterprise customer sales calls,
ranked by call frequency and evidence quality. call_count = distinct calls mentioning this theme.

IMPORTANT: Action Tracking and Accountability ranked #1 by frequency but was disqualified
after manual audit — all 22 member labels describe Voxel's existing safety workflow feature,
not a new non-safety opportunity. Do NOT include it in the top 3.

The corrected top 3 non-safety opportunities are:
1. Door Duration Monitoring (11 calls, 4.4/5 evidence)
2. Dashboard and UI Customization (8 calls, 5.0/5 evidence)
3. System Integration and Data Export (8 calls, 4.6/5 evidence)

Honorable mentions: Product Damage and Shrink Reduction (8 calls), Restricted Area and Access Control (7 calls).

Data for context:
{json.dumps(top_opps, indent=2)}

Also note: {len(noise_candidates)} files ({len(noise_candidates)/len(files)*100:.0f}% of corpus)
were pure admin/renewal calls with no product discussion and were excluded.

Write a tight business memo section for the CTO identifying the TOP 3 non-safety opportunities.

For each opportunity:
- Name it clearly
- State how many calls it appears in (as a fraction of the clean corpus, not total files)
- Make the business case in 1-2 sentences (who buys it, what pain it solves)
- Flag any uncertainty or caveat about the evidence quality

Then add ONE paragraph on the single most impactful pipeline improvement Voxel could make.

Tone: direct, no filler, written for a technical founder. No headers beyond the opportunity names.
Max 350 words.
"""

print("Generating business memo with corrected top 3...")
memo_response = client.messages.create(
    model=MODEL,
    max_tokens=600,
    messages=[{"role": "user", "content": memo_prompt}]
)

memo_text = memo_response.content[0].text
print(memo_text)

Generating business memo with corrected top 3...
## Top 3 Non-Safety Opportunities

**Door Duration Monitoring**
Mentioned in 11/95 calls (12%). Operations leaders want to track loading dock efficiency and turn-time — how long doors stay open correlates to throughput bottlenecks and energy waste. This is the clearest operational analytics play, though quotes are thin; we're inferring the full use case from brief mentions.

**Dashboard and UI Customization**
Mentioned in 8/95 calls (8%). Customers need shift-specific views and role-based boards to combat data overload — second shift supervisors want filtered dashboards showing only their cameras and use cases, not enterprise-wide noise. Quote evidence is strong (5.0/5), and this appears to be both a retention play for complex deployments and an expansion wedge into more cameras per site.

**System Integration and Data Export**
Mentioned in 8/95 calls (8%). Enterprise buyers want to pipe Voxel data into existing EHS platforms, BI tools, 

---
## 7. Export Results

Save the annotated DataFrame and cluster map for reference in your PR.

In [35]:
# Drop the list columns before saving (not CSV-friendly)
df_export = df.drop(columns=["quotes_list", "timestamps"], errors="ignore")
df_export.to_csv("annotated_use_cases.csv", index=False)

clusters_df.to_csv("label_clusters.csv", index=False)
opportunity_ranking.to_csv("opportunity_ranking.csv", index=False)

with open("memo_draft.txt", "w") as f:
    f.write(memo_text)

print("Saved:")
print("  annotated_use_cases.csv  — full df with all flags and scores")
print("  label_clusters.csv       — canonical label clusters")
print("  opportunity_ranking.csv  — ranked nonsafety opportunities")
print("  memo_draft.txt           — Claude-generated synthesis (review before using)")

Saved:
  annotated_use_cases.csv  — full df with all flags and scores
  label_clusters.csv       — canonical label clusters
  opportunity_ranking.csv  — ranked nonsafety opportunities
  memo_draft.txt           — Claude-generated synthesis (review before using)


---
## 8. Summary of Findings

This cell prints a full audit summary — useful to screenshot for your SUBMISSION.md.

In [65]:
clean_corpus = len(files) - len(noise_candidates)

print("=" * 60)
print("VOXEL PIPELINE AUDIT — FULL SUMMARY")
print("=" * 60)
print()
print("DATASET")
print(f"  Files analyzed       : {len(files)}")
print(f"  Total use cases      : {len(df)}")
print(f"  Safety               : {len(df[df.bucket == 'safety_use_cases'])}")
print(f"  Non-safety           : {len(df[df.bucket == 'nonsafety_use_cases'])}")
print()
print("QUALITY ISSUES FOUND")
print(f"  [Rule] Missing timestamps        : {len(schema_df)} evidence items")
print(f"  [Rule] Cross-bucket contamination: {len(contaminated_files)} files, {len(contaminated)} quotes")
print(f"  [Rule] Admin noise candidates    : {len(noise_candidates)} files ({len(noise_candidates)/len(files)*100:.0f}%)")
print(f"  [Rule] Recycled evidence         : {len(shared_evidence)} quotes across multiple labels")
print(f"  [Rule] Transcription artifacts   : {len(trans_df)} suspect quotes")
print(f"  [LLM]  Contaminated labels       : {clusters_df.contamination_flag.sum()} canonical themes")
print(f"  [LLM]  Generic/noise labels      : {clusters_df.generic_flag.sum()} canonical themes")
print(f"  [LLM]  Off-topic use cases       : {df.off_topic.sum()} rows")
print(f"  [LLM]  Avg evidence support      : {df.evidence_support.mean():.2f}/5")
print(f"  [LLM]  Avg label accuracy        : {df.label_accuracy.mean():.2f}/5")
print()
print("TOP 3 NON-SAFETY OPPORTUNITIES (post-audit corrected)")
print("  NOTE: Action Tracking (ranked #1 by frequency) was disqualified")
print("  after Layer 3 audit — all member labels describe safety workflow,")
print("  not a new non-safety opportunity.")
print()
corrected_top3 = [
    ("Door Duration Monitoring",         11, 4.4),
    ("Dashboard and UI Customization",   8,  5.0),
    ("System Integration and Data Export",8, 4.6),
]
for i, (name, calls, evidence) in enumerate(corrected_top3, 1):
    pct = calls / clean_corpus * 100
    print(f"  {i}. {name}")
    print(f"     Appears in {calls} calls ({pct:.0f}% of clean corpus), avg evidence: {evidence}/5")
print()
print("HONORABLE MENTIONS")
print("  Product Damage and Shrink Reduction  — 8 calls, new buyer persona (ops/finance)")
print("  Restricted Area and Access Control   — 7 calls, genuine loss prevention play")
print("=" * 60)

VOXEL PIPELINE AUDIT — FULL SUMMARY

DATASET
  Files analyzed       : 99
  Total use cases      : 702
  Safety               : 471
  Non-safety           : 231

QUALITY ISSUES FOUND
  [Rule] Missing timestamps        : 222 evidence items
  [Rule] Cross-bucket contamination: 48 files, 207 quotes
  [Rule] Admin noise candidates    : 4 files (4%)
  [Rule] Recycled evidence         : 20 quotes across multiple labels
  [Rule] Transcription artifacts   : 7 suspect quotes
  [LLM]  Contaminated labels       : 4 canonical themes
  [LLM]  Generic/noise labels      : 22 canonical themes
  [LLM]  Off-topic use cases       : 11 rows
  [LLM]  Avg evidence support      : 4.75/5
  [LLM]  Avg label accuracy        : 4.34/5

TOP 3 NON-SAFETY OPPORTUNITIES (post-audit corrected)
  NOTE: Action Tracking (ranked #1 by frequency) was disqualified
  after Layer 3 audit — all member labels describe safety workflow,
  not a new non-safety opportunity.

  1. Door Duration Monitoring
     Appears in 11 calls (12

---
## 9. Three-Layer Audit — Validating the LLM Output

Before writing the memo, we validate Claude's work against what we already
know from manual review. Zero API cost — all local.

**Layer 1** — known issue validation (label-level only, not cross-bucket)

**Layer 2** — human review of Claude's 22 generic flags

**Layer 3** — cluster consistency check (over-clustered groups, singletons, top 10 sanity check)

In [59]:
# ── LAYER 1: sanity check known issues from manual review ────────────────────
print("=" * 55)
print("LAYER 1 — Known Issues Validation")
print("=" * 55)
print("Note: cross-bucket contamination is tested in Step 2")
print("(rule-based). Layer 1 only tests label-level issues")
print("that Claude's clustering was responsible for.")
print()

known_issues = [
    # (label, expected_generic, note)
    ("Getting sites onto the same schedule",
     True, "Summit Span — pure admin scheduling talk, not a product use case"),
    ("Contract renewal language clarification",
     True, "Summit Span — contract obligations discussion, not a product use case"),
    ("Quarterly shareholder reporting",
     True, "Prairie Bridge — customer thanking for report, pure account management"),
    ("Reduce safety incidents and claims using AI video analytics",
     True, "Peak Ridge — customer explicitly rejected the product, not a use case"),
    ("Upgrade legacy NVR/VMS with AI capabilities",
     True, "Peak Ridge — about their existing VMS software, off-topic"),
    ("PIT-to-pedestrian proximity alerts",
     False, "Trail Lift Works — well evidenced, correctly in safety, should NOT be flagged"),
    ("Reduce property damage, equipment repair costs, and product shrink from PIT incidents",
     False, "Trail Lift Works — genuine nonsafety opp, should NOT be flagged"),
    ("Ergonomic risk detection (improper bends and overreach)",
     False, "Voxel Demo — 6 strong quotes, correctly in safety, should NOT be flagged"),
]

passed = 0
failed = 0

for label, exp_generic, note in known_issues:
    row = df[df.label == label]
    if len(row) == 0:
        print(f"  NOT FOUND : {label[:60]}")
        print(f"              ({note})")
        failed += 1
        continue

    got_generic = bool(row.generic_flag.values[0])
    g_pass      = got_generic == exp_generic
    status      = "PASS" if g_pass else "FAIL"

    if status == "PASS":
        passed += 1
    else:
        failed += 1

    symbol = "✓" if status == "PASS" else "✗"
    print(f"  {symbol} {status} : {label[:60]}")
    print(f"         Generic flag expected: {exp_generic}, got: {got_generic} {'OK' if g_pass else 'WRONG'}")
    print(f"         ({note})")
    print()

total    = len(known_issues)
true_pos = sum(1 for l, e, n in known_issues if e == True)
true_neg = sum(1 for l, e, n in known_issues if e == False)

noise_passed = sum(
    1 for l, e, n in known_issues
    if e == True
    and len(df[df.label == l]) > 0
    and bool(df[df.label == l].generic_flag.values[0]) == True
)

clean_passed = sum(
    1 for l, e, n in known_issues
    if e == False
    and len(df[df.label == l]) > 0
    and bool(df[df.label == l].generic_flag.values[0]) == False
)

print(f"Result: {passed}/{total} correctly flagged")
print()
print(f"  Noise cases Claude should flag    : {true_pos} (caught {noise_passed})")
print(f"  Clean cases Claude should NOT flag: {true_neg} (correctly left {clean_passed} unflagged)")
print()
print(f"Memo line: Claude's generic flagging caught {noise_passed}/{true_pos} real noise")
print(f"cases and correctly left {clean_passed}/{true_neg} clean cases unflagged.")

LAYER 1 — Known Issues Validation
Note: cross-bucket contamination is tested in Step 2
(rule-based). Layer 1 only tests label-level issues
that Claude's clustering was responsible for.

  ✓ PASS : Getting sites onto the same schedule
         Generic flag expected: True, got: True OK
         (Summit Span — pure admin scheduling talk, not a product use case)

  ✓ PASS : Contract renewal language clarification
         Generic flag expected: True, got: True OK
         (Summit Span — contract obligations discussion, not a product use case)

  ✗ FAIL : Quarterly shareholder reporting
         Generic flag expected: True, got: False WRONG
         (Prairie Bridge — customer thanking for report, pure account management)

  ✗ FAIL : Reduce safety incidents and claims using AI video analytics
         Generic flag expected: True, got: False WRONG
         (Peak Ridge — customer explicitly rejected the product, not a use case)

  ✗ FAIL : Upgrade legacy NVR/VMS with AI capabilities
         G

Why Layer 1 Cross-Bucket Testing Was Not a Fair Test
We tested whether Claude's label clustering caught cross-bucket contamination. It caught 0 out of 8 cases we identified manually. That sounds bad but the test was set up wrong.
Claude's clustering job was to group similar labels together across all files. It was never asked to compare safety and nonsafety buckets within the same file. We split the clustering into two separate API calls — one for safety labels, one for nonsafety labels — so Claude never saw both sides at the same time. Expecting it to catch cross-bucket contamination under those conditions is like asking someone to spot a duplicate between two documents they were never shown together.
The cross-bucket contamination was already caught correctly in Step 2 using rule-based quote matching. That found 207 contaminated quotes across 48 files. That is the right tool for that job. Claude's clustering was the wrong tool.
The 2 admin noise passes (Summit Span cases) were valid tests because those are label-level problems Claude could actually see. The 6 cross-bucket failures were testing something Claude was architecturally prevented from doing.

---
### Layer 2 — Human Review of Generic Flags

Pull all 22 generic-flagged clusters from clusters_df and display for human review.
Then log disagreements — cases where Claude wrongly flagged a real product signal as noise.

In [47]:
# ── Layer 2: pull generic flags directly from data ────────────────────────────
generic_flagged = clusters_df[
    clusters_df.generic_flag == True
][["canonical_name", "bucket", "generic_reason"]].reset_index(drop=True)

generic_flagged.index += 1
generic_flagged["bucket"] = generic_flagged["bucket"].str.replace(
    "_use_cases", ""
).str.upper()

# print as clean readable text — no styling issues, no hardcoding
print(f"Claude flagged {len(generic_flagged)} clusters as generic/noise\n")
print(f"{'#':<4} {'Bucket':<12} {'Canonical Name':<45} Claude's Reason")
print("-" * 120)
for i, row in generic_flagged.iterrows():
    name   = row["canonical_name"][:43]
    bucket = row["bucket"][:10]
    reason = row["generic_reason"] if row["generic_reason"] else "no reason given"
    print(f"{i:<4} {bucket:<12} {name:<45} {reason}")

Claude flagged 22 clusters as generic/noise

#    Bucket       Canonical Name                                Claude's Reason
------------------------------------------------------------------------------------------------------------------------
1    SAFETY       Drone-Based Remote Inspection                 This refers to a separate technology (drones) not directly related to Voxel's core computer vision platform.
2    SAFETY       Fatigue Monitoring Inquiry                    This label explicitly states the capability is not supported by Voxel.
3    SAFETY       EHS Management Software Adoption              This is a generic reference to adopting EHS software, not a specific Voxel safety use case.
4    SAFETY       Camera Enhancements for Safety                This is a vague reference to camera enhancements similar to Voxel rather than a concrete use case.
5    SAFETY       Machine Shutdown Use Case                     This is too vague and does not specify what safety behavior is 

In [48]:
# ── your verdicts (only fill in ones you have an opinion on) ──────────────────
my_disagreements = [
    # (row_number_from_table, verdict,    your_reason)
    (3,  "DISAGREE", "EHS integration is a real product opportunity"),
    (4,  "PARTIAL",  "Product feedback, not pure noise"),
    (5,  "DISAGREE", "Machine shutdown is a real Voxel use case"),
    (6,  "PARTIAL",  "Vague but not noise — still safety adjacent"),
]

# ── scoring — all pulled from clusters_df, nothing hardcoded ─────────────────
generic_flagged = clusters_df[
    clusters_df.generic_flag == True
][["canonical_name", "bucket", "generic_reason"]].reset_index(drop=True)
generic_flagged.index += 1

total    = len(generic_flagged)
wrong    = sum(1 for d in my_disagreements if d[1] == "DISAGREE")
partial  = sum(1 for d in my_disagreements if d[1] == "PARTIAL")
correct  = total - wrong - partial

print(f"Layer 2 Generic Flag Audit — {total} clusters reviewed")
print(f"  AGREE    (Claude correct) : {correct} ({correct/total*100:.0f}%)")
print(f"  DISAGREE (Claude wrong)   : {wrong}   ({wrong/total*100:.0f}%)")
print(f"  PARTIAL  (borderline)     : {partial}  ({partial/total*100:.0f}%)")
print()
print(f"Claude's generic flagging accuracy: {correct}/{total} = {correct/total*100:.0f}%")

if my_disagreements:
    print("\nWrongly flagged — real signals excluded from ranking:")
    for num, verdict, reason in my_disagreements:
        name = generic_flagged.loc[num, "canonical_name"]
        print(f"  [{verdict}] #{num} {name}")
        print(f"           Reason: {reason}")

print(f"\nMemo line: Claude's generic flagging was {correct/total*100:.0f}% accurate")
print(f"on manual review. {wrong} real signals wrongly excluded from opportunity ranking.")

Layer 2 Generic Flag Audit — 22 clusters reviewed
  AGREE    (Claude correct) : 18 (82%)
  DISAGREE (Claude wrong)   : 2   (9%)
  PARTIAL  (borderline)     : 2  (9%)

Claude's generic flagging accuracy: 18/22 = 82%

Wrongly flagged — real signals excluded from ranking:
  [DISAGREE] #3 EHS Management Software Adoption
           Reason: EHS integration is a real product opportunity
  [PARTIAL] #4 Camera Enhancements for Safety
           Reason: Product feedback, not pure noise
  [DISAGREE] #5 Machine Shutdown Use Case
           Reason: Machine shutdown is a real Voxel use case
  [PARTIAL] #6 Powered Equipment Safety (Generic)
           Reason: Vague but not noise — still safety adjacent

Memo line: Claude's generic flagging was 82% accurate
on manual review. 2 real signals wrongly excluded from opportunity ranking.


---
### Layer 3 — Cluster Consistency Check

Check whether Claude grouped labels at a sensible level of detail.
Part 1: scrollable view of over-clustered groups (15+ members).
Part 2: human verdicts on whether those groups are too broad.
Part 3: singleton rate check.
Part 4: top 10 opportunities — full member label sanity check.

In [57]:
from IPython.display import HTML

def make_scrollable_clusters(clusters_df, threshold=15):
    over = clusters_df[
        clusters_df.member_count >= threshold
    ].sort_values("member_count", ascending=False).reset_index(drop=True)
    
    html = """
    <div style="height:600px; overflow-y:scroll; border:1px solid #ccc; 
                padding:12px; font-family:monospace; font-size:12px;">
    """
    
    for _, row in over.iterrows():
        bucket_color = "#2563EB" if "safety" in row["bucket"] else "#16A34A"
        html += f"""
        <div style="margin-bottom:20px; border-left:3px solid {bucket_color}; 
                    padding-left:10px;">
            <div style="font-weight:bold; font-size:13px;">
                {row['canonical_name']} 
                <span style="color:{bucket_color}; font-size:11px;">
                    [{row['bucket'].replace('_use_cases','').upper()}]
                </span>
                <span style="color:#888; font-size:11px;">
                    — {row['member_count']} members
                </span>
            </div>
            <div style="margin-top:6px; color:#444;">
        """
        for lbl in row["member_labels"]:
            html += f"&nbsp;&nbsp;— {lbl}<br>"
        
        html += "</div></div>"
    
    html += "</div>"
    return HTML(html)

make_scrollable_clusters(clusters_df, threshold=15)

---
### Layer 3 — Over-Broad Cluster Verdicts

**Actions Workflow and Coaching — OVER_BROAD**

26 safety members overlap heavily with the nonsafety Action Tracking cluster. 
Same concept split across two buckets causing double-counting.

**Action Tracking and Accountability — OVER_BROAD**

Ranked #1 nonsafety opportunity but 22 members contain explicitly safety-labeled content. 
This is cross-bucket contamination at the cluster level. Not a genuine new business opportunity.

In [53]:
from IPython.display import HTML

singletons = clusters_df[
    clusters_df.member_count == 1
][["canonical_name", "bucket"]].reset_index(drop=True)
singletons.index += 1

rows = ""
for i, row in singletons.iterrows():
    bucket = row["bucket"].replace("_use_cases", "").upper()
    rows += f"<tr><td>{i}</td><td>{bucket}</td><td>{row['canonical_name']}</td></tr>"

display(HTML(f"""
<p><b>Singletons: {len(singletons)} / {len(clusters_df)} clusters ({len(singletons)/len(clusters_df)*100:.0f}%)</b></p>
<div style="height:400px; overflow-y:scroll;">
  <table border="1" cellpadding="6" style="border-collapse:collapse; font-size:12px; width:100%;">
    <thead><tr><th>#</th><th>Bucket</th><th>Canonical Name</th></tr></thead>
    <tbody>{rows}</tbody>
  </table>
</div>
"""))

#,Bucket,Canonical Name
1,SAFETY,Forklift Seatbelt Compliance
2,SAFETY,Fire Door and Dock Door Monitoring
3,SAFETY,Forklift Pre-Use Inspection Compliance
4,SAFETY,Signage and Traffic Control Analytics
5,SAFETY,Daily Safety Digest and Reporting
6,SAFETY,Facility Layout and Engineering Controls
7,SAFETY,Ladder Safety Monitoring
8,SAFETY,Employee vs Customer Distinction
9,SAFETY,Data Export and Integration
10,SAFETY,Line-of-Fire Hazard Detection


In [55]:
from IPython.display import HTML

rows = ""
for rank, (_, opp) in enumerate(opportunity_ranking.head(10).iterrows(), 1):
    cluster_row = clusters_df[clusters_df.canonical_name == opp["canonical"]]
    member_count = cluster_row["member_count"].values[0] if len(cluster_row) else "?"
    all_labels = cluster_row["member_labels"].values[0] if len(cluster_row) else []
    
    labels_html = "".join([f"<div>— {lbl}</div>" for lbl in all_labels])
    
    rows += f"""
    <tr>
      <td style="padding:8px; font-weight:bold; text-align:center;">#{rank}</td>
      <td style="padding:8px; font-weight:bold;">{opp['canonical']}</td>
      <td style="padding:8px; text-align:center;">{opp['call_count']}</td>
      <td style="padding:8px; text-align:center;">{opp['avg_evidence_support']:.1f}/5</td>
      <td style="padding:8px; text-align:center;">{member_count}</td>
      <td style="padding:8px; font-size:11px; color:#555;">{labels_html}</td>
    </tr>
    """

display(HTML(f"""
<div style="height:600px; overflow-y:scroll;">
  <table border="1" cellpadding="4" 
         style="border-collapse:collapse; font-size:12px; width:100%;">
    <thead style="position:sticky; top:0; background:#1B2A4A; color:white;">
      <tr>
        <th style="padding:8px;">Rank</th>
        <th style="padding:8px;">Canonical Theme</th>
        <th style="padding:8px;"># Calls</th>
        <th style="padding:8px;">Evidence</th>
        <th style="padding:8px;">Members</th>
        <th style="padding:8px;">All Labels Grouped Here</th>
      </tr>
    </thead>
    <tbody>{rows}</tbody>
  </table>
</div>
"""))

Rank,Canonical Theme,# Calls,Evidence,Members,All Labels Grouped Here
#1,Action Tracking and Accountability,20,4.9/5,22,"— Action workflow and accountability tracking— Track and find completed/in-progress actions across Wilmington— Reminders for past-due actions— Assign and track actions with accountability and reporting— Actions workflow for incident and task management— Automated reminders for overdue actions— Accountable coaching and training via Actions tied to incident clips— Safety tallying and follow-up workflow (tracking repeat offenders/situations)— Action management workflow with ownership, cross-shift follow-up, timelines, and impact markers— Supervisor accountability and action tracking in Voxel— Assign and track safety actions with role-based access— Track and close the loop on safety corrective actions with the Actions workflow— Action tracking and accountability for safety interventions— Assign and track follow-ups with Actions— Assign and track corrective actions in platform— Voxel-assigned actions to drive engagement and accountability— Assign incidents and track accountability— Action management and coaching workflow— Use Actions workflow to manage corrective actions and track impact— Assign and track actions with owners and due dates— Clarify workflow for rejected assigned actions and reassignment— Show overdue/upcoming actions and allow due date edits"
#2,Door Duration Monitoring,11,4.4/5,10,— Operational analytics: monitor open door durations— Open cooler/freezer door monitoring for energy/product protection— Dock door open-duration tracking— Open dock door duration monitoring and threshold tuning— Door open/propped duration monitoring— Open door duration monitoring— Open door monitoring for security— Cold storage door monitoring for energy savings— Open door duration tracking for turn-time management— After-hours dock door open alert (lights off)
#3,Dashboard and UI Customization,8,5.0/5,10,"— Restore dashboard deep links in daily incident email— Role‑based boards and dashboard customization for MODs— Executive all-sites dashboard with custom date ranges for weekly leadership reporting— Shift-based dashboards and gamification— Role-based boards and camera access for supervisors— Use of Voxel boards for simple data visualization and team communication— Distributed area ownership via Boards and tier process— Role/shift/incident-specific dashboards (""boards"") for operations— Data overload management via Boards and filtered default views— Boards for targeted monitoring and trend detection"
#4,Safety Analytics and Reporting,8,5.0/5,10,"— Safety performance analytics, gamification, and automated reporting— Year-over-year incident trend reporting and comparison— Safety incident reporting and trend analysis— Safety performance benchmarking and goal tracking— Weekly trend reporting and executive/corporate views— Safety program analytics and workflow management— Safety analytics and interactive reporting across shops— Enterprise safety analytics and summary view— Safety reporting and coaching workflows— Cross-site and shift compliance benchmarking"
#5,System Integration and Data Export,8,4.6/5,8,— Integration with incident-management and reporting systems— Export operational data to external BI/EHS systems— Integrate with existing camera infrastructure (no hardware replacement)— Data export/API integration for custom analytics— API and BI integration for leading/lagging metrics— Integration with an external workflow tool via API/data export— API feed updates/integration— Integrate with existing cameras and minimize IT burden
#6,Product Damage and Shrink Reduction,8,4.2/5,9,"— Reduce shrink by detecting product damage from PIT impacts— Product and infrastructure damage detection (collisions with racks/product)— Product condition monitoring— Sensitive product condition monitoring— Product damage tracking— Product condition and dwell-time detection— Reduce property damage, equipment repair costs, and product shrink 

## Findings:
#1 Action Tracking and Accountability — Disqualified
All 22 labels are about Voxel's existing Actions safety workflow. Not a new opportunity. Dropped.

#2 Door Duration Monitoring — Valid
Clean signal, all door duration variations. Real #1.

#3 Dashboard and UI Customization — Valid
Role-based boards, shift dashboards, gamification. Genuine ops customization. Real #2.

#4 Safety Analytics and Reporting — Borderline
Analytics on top of safety data. Extension of core product, not a new opportunity.

#5 System Integration and Data Export — Valid
API feeds, BI/EHS integration, data export. Real #3.

#6 Product Damage and Shrink Reduction — Valid
Clean shrink/damage signal, new buyer persona. Honorable mention.

#7 Restricted Area and Access Control — Valid
Loss prevention, after-hours monitoring. Genuinely non-safety. Honorable mention.

#8 Warehouse Productivity Optimization — Mixed
One wearables label slipped in. Rest are valid.

#9 Report Export and Subscriptions — Mixed
Shareholder reporting (admin noise) mixed in with legitimate reporting features.

#10 Intervention Impact Tracking — Borderline
Measuring safety outcomes. Adjacent to core product, not a standalone opportunity.

---
### Layer 3 — Part 4 Findings Summary

**#1 Action Tracking and Accountability** — Disqualified. All 22 labels are safety workflow. Dropped.

**#2 Door Duration Monitoring** — Valid. Clean signal. Real #1.

**#3 Dashboard and UI Customization** — Valid. Role-based boards, gamification. Real #2.

**#4 Safety Analytics and Reporting** — Borderline. Extension of core product, not a new opportunity.

**#5 System Integration and Data Export** — Valid. API/BI integration play. Real #3.

**#6 Product Damage and Shrink Reduction** — Valid. New buyer persona. Honorable mention.

**#7 Restricted Area and Access Control** — Valid. Loss prevention play. Honorable mention.

**#8 Warehouse Productivity Optimization** — Mixed. One wearables label slipped in.

**#9 Report Export and Subscriptions** — Mixed. Shareholder reporting noise mixed in.

**#10 Intervention Impact Tracking** — Borderline. Safety outcome measurement, adjacent to core product.

---
**Final top 3:** Door Duration Monitoring (11 calls), Dashboard and UI Customization (8 calls), System Integration and Data Export (8 calls).